# Notebook 58 — Agent Trust and Provenance

Demonstrates `multigen.agent_trust`: HMAC-SHA256 agent attestation, an append-only
capability ledger, a Merkle-linked workflow provenance chain, invisible zero-width
Unicode watermarking, and composite trust scoring.
No live LLM or external service is required — everything runs locally.

## 1. Setup

In [ ]:
import sys
import importlib.util

sys.path.insert(0, '../sdk')

def load(name):
    spec = importlib.util.spec_from_file_location(
        f'multigen.{name}', f'../sdk/multigen/{name}.py')
    mod = importlib.util.module_from_spec(spec)
    sys.modules[f'multigen.{name}'] = mod
    spec.loader.exec_module(mod)
    return mod

at = load('agent_trust')

from multigen.agent_trust import (
    AgentTrustManager,
    AgentCapability,
    AgentAttestor,
    AgentIdentity,
    CapabilityLedger,
    LedgerEntry,
    WorkflowProvenanceChain,
    ProvenanceRecord,
    OutputWatermarker,
    WatermarkPayload,
    TrustScorer,
    TrustScore,
)

# Create a shared manager with a fixed HMAC secret (never do this in production)
mgr = AgentTrustManager(secret='demo-secret-key-42')
print('AgentTrustManager initialised OK')
print('  attestor   :', type(mgr.attestor).__name__)
print('  ledger     :', type(mgr.ledger).__name__)
print('  chain      :', type(mgr.chain).__name__)
print('  watermarker:', type(mgr.watermarker).__name__)
print('  scorer     :', type(mgr.scorer).__name__)

## 2. Agent Attestation

`AgentAttestor` computes a SHA-256 digest of the agent class source code at
registration time and binds it to a stable `AgentIdentity` via HMAC-SHA256.
Re-verifying later detects any source-level tampering.

In [ ]:
# Define a simple dummy agent class whose source is inspectable
class SummaryAgent:
    """A dummy agent that summarises text."""

    def __init__(self, name: str = 'summary-agent'):
        self.name = name

    def run(self, text: str) -> str:
        words = text.split()
        return ' '.join(words[:10]) + ('...' if len(words) > 10 else '')


agent_obj = SummaryAgent()
print('Agent class defined:', type(agent_obj).__name__)

In [ ]:
# Attest the agent — returns an AgentIdentity
agent_id_obj: AgentIdentity = mgr.attestor.attest(agent_obj, version='1.0.0')
agent_id = agent_id_obj.agent_id

print('AgentIdentity fields:')
print(f'  agent_id   : {agent_id[:24]}...  (HMAC-derived, truncated for display)')
print(f'  name       : {agent_id_obj.name}')
print(f'  version    : {agent_id_obj.version}')
print(f'  code_hash  : {agent_id_obj.code_hash[:24]}...  (SHA-256 of source)')
print(f'  created_at : {agent_id_obj.created_at:.3f}')

In [ ]:
# Verify the unmodified agent — should return True
result = mgr.attestor.verify(agent_id, agent_obj)
print('verify(agent_id, original_agent) ->', result)  # True
assert result is True, 'Verification of the original agent must succeed'

In [ ]:
# Simulate source tampering: define a NEW class with different source
class SummaryAgent:  # same name, different body — source hash changes
    """A tampered version of SummaryAgent."""

    def __init__(self, name: str = 'summary-agent'):
        self.name = name

    def run(self, text: str) -> str:
        # malicious logic injected here
        return 'TAMPERED OUTPUT'


tampered_agent = SummaryAgent()
result_tampered = mgr.attestor.verify(agent_id, tampered_agent)
print('verify(agent_id, tampered_agent) ->', result_tampered)  # False
assert result_tampered is False, 'Verification of the tampered agent must fail'

## 3. Capability Ledger

`CapabilityLedger` is append-only: every grant and revoke action is recorded
as an immutable `LedgerEntry`.  The effective permission set is always derivable
by replaying the log — nothing is deleted.

In [ ]:
# Grant three capabilities to the agent
cap_search = AgentCapability('web_search', 'Search the public web', 'medium')
cap_db     = AgentCapability('database_write', 'Write to the production database', 'high')
cap_log    = AgentCapability('audit_log_read', 'Read audit log entries', 'low')

mgr.ledger.grant(agent_id, cap_search, 'admin')
mgr.ledger.grant(agent_id, cap_db,     'admin')
mgr.ledger.grant(agent_id, cap_log,    'admin')

print('Capabilities granted:')
for name in ['web_search', 'database_write', 'audit_log_read']:
    status = 'ACTIVE' if mgr.ledger.has(agent_id, name) else 'INACTIVE'
    print(f'  {name:20s}  {status}')

In [ ]:
# has() queries the current effective state
print('has(web_search)      ->', mgr.ledger.has(agent_id, 'web_search'))       # True
print('has(database_write)  ->', mgr.ledger.has(agent_id, 'database_write'))   # True
print('has(send_email)      ->', mgr.ledger.has(agent_id, 'send_email'))       # False (never granted)

In [ ]:
# Revoke a high-risk capability
mgr.ledger.revoke(agent_id, 'database_write', revoked_by='security-team')
print('After revoke:')
print('  has(database_write) ->', mgr.ledger.has(agent_id, 'database_write'))  # False
print('  has(web_search)     ->', mgr.ledger.has(agent_id, 'web_search'))      # True (not revoked)

# Attempting to revoke a capability that is not active raises KeyError
try:
    mgr.ledger.revoke(agent_id, 'database_write', revoked_by='security-team')
except KeyError as exc:
    print('KeyError (double-revoke):', exc)

In [ ]:
# Show the full immutable audit trail
trail = mgr.ledger.audit_trail(agent_id)
print(f'Audit trail ({len(trail)} entries):')
print(f'  {"entry_id":10s}  {"action":6s}  {"capability":20s}  {"risk":8s}  {"actor"}')
print('  ' + '-' * 70)
for e in trail:
    eid_short = e.entry_id[:8] + '...'
    print(f'  {eid_short:10s}  {e.action:6s}  {e.capability.name:20s}  {e.capability.risk_level:8s}  {e.actor}')

## 4. Workflow Provenance Chain

`WorkflowProvenanceChain` maintains a Merkle-linked, HMAC-signed sequence of
provenance records.  Each record hashes its input and output data and includes
a `parent_id` link so the chain can be walked back to the root.
`verify_chain()` re-derives every HMAC signature to detect tampering.

In [ ]:
# Record 3 sequential workflow operations with parent links
rec1 = mgr.chain.record(
    agent_id=agent_id,
    operation='fetch_data',
    input_data={'source': 'crm_database', 'query': 'SELECT * FROM leads LIMIT 100'},
    output_data={'rows': 100, 'schema': ['id', 'name', 'email', 'score']},
    parent_id=None,  # root record
)

rec2 = mgr.chain.record(
    agent_id=agent_id,
    operation='enrich_data',
    input_data={'rows': 100, 'schema': ['id', 'name', 'email', 'score']},
    output_data={'rows': 100, 'enriched_fields': ['industry', 'company_size']},
    parent_id=rec1.record_id,  # linked to rec1
)

rec3 = mgr.chain.record(
    agent_id=agent_id,
    operation='generate_report',
    input_data={'rows': 100, 'enriched_fields': ['industry', 'company_size']},
    output_data={'report': 'Q3 Lead Enrichment Summary', 'pages': 4},
    parent_id=rec2.record_id,  # linked to rec2
)

print('Recorded 3 provenance steps:')
for i, rec in enumerate([rec1, rec2, rec3], 1):
    pid = rec.parent_id[:12] + '...' if rec.parent_id else 'None (root)'
    print(f'  Step {i}: {rec.operation:20s}  id={rec.record_id[:12]}...  parent={pid}')

In [ ]:
# verify_chain() re-derives all HMAC signatures and checks parent links
chain_ok = mgr.chain.verify_chain()
print('verify_chain() ->', chain_ok)  # True
assert chain_ok, 'Chain verification must pass for an unmodified chain'

In [ ]:
# get_lineage() walks from a given record back to the root
lineage = mgr.chain.get_lineage(rec3.record_id)
print(f'Lineage from rec3 to root ({len(lineage)} records, newest first):')
for step, rec in enumerate(lineage):
    parent_label = rec.parent_id[:12] + '...' if rec.parent_id else 'None  <-- root'
    print(f'  [{step}] operation={rec.operation:20s}  parent={parent_label}')

## 5. Zero-Width Watermarking

`OutputWatermarker` encodes a random `watermark_id` as invisible zero-width
Unicode characters (U+200B, U+200C, U+200D) injected immediately after the
first word of the text.  The watermarked content looks identical to the
original in any standard renderer.

In [ ]:
original_text = 'This is agent output text.'

payload: WatermarkPayload = mgr.watermarker.watermark(original_text, agent_id)

print('Original text :', repr(original_text))
print()
print('Watermarked   :', repr(payload.content))
print()
print('Visually identical? ->', original_text.replace('\u200b','').replace('\u200c','').replace('\u200d','') == payload.content.replace('\u200b','').replace('\u200c','').replace('\u200d',''))
print()
print('watermark_id  :', payload.watermark_id)
print('agent_id (pfx):', payload.agent_id[:24] + '...')
print('timestamp     :', payload.timestamp)

In [ ]:
# Show the hidden character count — demonstrating the watermark is purely invisible
visible_chars  = [c for c in payload.content if c not in ('\u200b', '\u200c', '\u200d')]
hidden_chars   = [c for c in payload.content if c     in ('\u200b', '\u200c', '\u200d')]

print(f'Total chars in payload.content : {len(payload.content)}')
print(f'Visible characters             : {len(visible_chars)}')
print(f'Hidden zero-width characters   : {len(hidden_chars)}')
print()
# When rendered normally, the visible text is unchanged
print('Rendered (invisible chars stripped):', ''.join(visible_chars))

In [ ]:
# extract() recovers the watermark_id from the text alone
extracted_id = mgr.watermarker.extract(payload.content)
print('Extracted watermark_id:', extracted_id)
print('Matches payload.watermark_id:', extracted_id == payload.watermark_id)

In [ ]:
# verify() checks the extracted id against payload.watermark_id via HMAC compare
verified = mgr.watermarker.verify(payload)
print('watermarker.verify(payload) ->', verified)  # True
assert verified, 'Watermark verification must pass for an intact payload'

# Verify fails if the content is stripped of hidden characters
from dataclasses import replace
stripped_payload = WatermarkPayload(
    content=''.join(visible_chars),  # zero-width chars removed
    watermark_id=payload.watermark_id,
    agent_id=payload.agent_id,
    timestamp=payload.timestamp,
)
print('verify(stripped_payload) ->', mgr.watermarker.verify(stripped_payload))  # False

## 6. Trust Scoring

`TrustScorer.score()` computes a weighted composite score in **[0, 1]** from
four independent signals:

| Component            | Weight | Signal source                         |
|----------------------|--------|---------------------------------------|
| attestation_valid    | 0.30   | AgentAttestor registry                |
| capability_compliance| 0.30   | CapabilityLedger (risk-level penalty) |
| provenance_intact    | 0.20   | WorkflowProvenanceChain.verify_chain()|
| behavioral_history   | 0.20   | Fraction of valid HMAC records        |

In [ ]:
trust: TrustScore = mgr.scorer.score(agent_id, mgr.attestor, mgr.ledger, mgr.chain)

print('TrustScore components:')
weights = {'attestation_valid': 0.30, 'capability_compliance': 0.30,
           'provenance_intact': 0.20, 'behavioral_history': 0.20}
for component, raw_score in trust.components.items():
    weighted = raw_score * weights[component]
    print(f'  {component:24s}  raw={raw_score:.4f}  weight={weights[component]:.2f}  contribution={weighted:.4f}')

print()
print(f'Overall trust score: {trust.score:.6f}  (max=1.0)')
print(f'Computed at        : {trust.computed_at:.3f}')

In [ ]:
# Show how adding a critical-risk capability lowers the score
cap_critical = AgentCapability('execute_arbitrary_code', 'Run any Python on host', 'critical')
mgr.ledger.grant(agent_id, cap_critical, 'rogue-actor')

trust_degraded: TrustScore = mgr.scorer.score(agent_id, mgr.attestor, mgr.ledger, mgr.chain)

print('After granting a CRITICAL capability:')
print(f'  capability_compliance : {trust_degraded.components["capability_compliance"]:.4f}')
print(f'  Overall trust score   : {trust_degraded.score:.6f}  (was {trust.score:.6f})')

# Revoke it and re-score to show recovery
mgr.ledger.revoke(agent_id, 'execute_arbitrary_code', 'security-team')
trust_recovered: TrustScore = mgr.scorer.score(agent_id, mgr.attestor, mgr.ledger, mgr.chain)
print(f'  After revoke          : {trust_recovered.score:.6f}')

## 7. Full Pipeline

End-to-end demonstration: attest an agent, grant capabilities, record
two provenance steps, watermark the output, and compute a final trust score.
A summary table is printed at the end.

In [ ]:
# Fresh manager for a clean pipeline
pipe = AgentTrustManager(secret='pipeline-demo-secret')

# --- Step 1: Define and attest agent ---
class ResearchAgent:
    """Agent that searches and summarises research papers."""

    def run(self, query: str) -> str:
        return f'Summary of results for: {query}'


research_agent = ResearchAgent()
identity = pipe.attestor.attest(research_agent, version='2.1.0')
rid = identity.agent_id
print(f'[1] Agent attested  id={rid[:16]}...')

In [ ]:
# --- Step 2: Grant capabilities ---
pipe.ledger.grant(rid, AgentCapability('web_search',     'Search the public web',     'medium'), 'admin')
pipe.ledger.grant(rid, AgentCapability('paper_database', 'Query research paper DB',   'low'),    'admin')
print(f'[2] Capabilities granted: web_search={pipe.ledger.has(rid, "web_search")}',
      f' paper_database={pipe.ledger.has(rid, "paper_database")}')

In [ ]:
# --- Step 3: Record provenance for 2 workflow steps ---
p_rec1 = pipe.chain.record(
    agent_id=rid,
    operation='search_papers',
    input_data={'query': 'multi-agent trust protocols 2024'},
    output_data={'hits': 42, 'top_paper': 'TrustChain: Verified Agent Collaboration'},
    parent_id=None,
)
p_rec2 = pipe.chain.record(
    agent_id=rid,
    operation='summarise_papers',
    input_data={'hits': 42, 'top_paper': 'TrustChain: Verified Agent Collaboration'},
    output_data={'summary': 'Recent work shows HMAC-linked provenance chains significantly ...'},
    parent_id=p_rec1.record_id,
)
chain_valid = pipe.chain.verify_chain()
print(f'[3] Provenance recorded: {len([p_rec1, p_rec2])} steps  chain_valid={chain_valid}')

In [ ]:
# --- Step 4: Watermark the agent output ---
output_text = 'Recent work shows HMAC-linked provenance chains significantly reduce impersonation risk in multi-agent systems.'
wm_payload = pipe.watermarker.watermark(output_text, rid)
wm_ok = pipe.watermarker.verify(wm_payload)
extracted = pipe.watermarker.extract(wm_payload.content)
print(f'[4] Output watermarked  verify={wm_ok}  extracted_id={extracted}')

In [ ]:
# --- Step 5: Compute trust score ---
final_trust = pipe.trust_score(rid)
print(f'[5] Trust score computed: {final_trust.score:.6f}')

In [ ]:
# --- Summary table ---
print()
print('=' * 65)
print('  AGENT TRUST PIPELINE — SUMMARY')
print('=' * 65)
print(f'  Agent class            : {type(research_agent).__name__}')
print(f'  Agent ID (prefix)      : {rid[:24]}...')
print(f'  Code hash (prefix)     : {identity.code_hash[:24]}...')
print(f'  Attestation valid      : {pipe.attestor.verify(rid, research_agent)}')
print()
active_caps = [e.capability.name for e in pipe.ledger.audit_trail(rid) if e.action == 'grant']
# derive current active set
current_caps = [name for name in set(active_caps) if pipe.ledger.has(rid, name)]
print(f'  Active capabilities    : {current_caps}')
print(f'  Ledger entries         : {len(pipe.ledger.audit_trail(rid))}')
print()
print(f'  Provenance records     : {len(pipe.chain._insertion_order)}')
print(f'  Chain integrity        : {pipe.chain.verify_chain()}')
lineage = pipe.chain.get_lineage(p_rec2.record_id)
print(f'  Lineage depth (rec2)   : {len(lineage)} records')
print()
print(f'  Watermark verified     : {wm_ok}')
print(f'  Watermark ID           : {extracted}')
print()
print('  Trust Score Breakdown:')
w = {'attestation_valid': 0.30, 'capability_compliance': 0.30,
     'provenance_intact': 0.20, 'behavioral_history': 0.20}
for k, v in final_trust.components.items():
    bar = '#' * int(v * 20)
    print(f'    {k:24s}  {v:.4f}  [{bar:<20s}]  weight={w[k]:.2f}')
print()
print(f'  OVERALL TRUST SCORE    : {final_trust.score:.6f} / 1.000000')
print('=' * 65)